In [ ]:
from pathlib import Path

import numpy as np


# Paths
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Reproducibility
RANDOM_STATE = 42

# Target / split
TARGET_COL = "y"
SESSION_COL = "session_id"
N_SPLITS_INNER = 5

# ============================================================
# Predictor groups
# ============================================================
NUMERIC_COLS = [
    "X", "Y",
    "above_ground_numeric",
    "Number of sighters", "Number of Squirrels", "Total Time of Sighting",
    "temperature",
    "squirrel_density_proxy",
]

# ============================================================
OHE_COLS = [
    "Shift",
    "Age",
    "Primary Fur Color",
    "Litter",
    "weather_condition",
]

MULTILABEL_COLS = [
    "Hectare Conditions",
]

CATEGORICAL_COLS = OHE_COLS + MULTILABEL_COLS

CATEGORY_DICT = {
    # OHE features
    "Shift": ["AM", "PM"],
    "Age": ["Adult", "Juvenile"],
    "Primary Fur Color": ["Black", "Cinnamon", "Gray"],
    "Litter": ["Abundant", "None", "Some", "Unknown"],
    "weather_condition": ["cloudy", "rainy", "sunny", "unknown"],
    # Multi-label features
    "Hectare Conditions": ["Busy", "Calm", "Moderate"],
}

# ============================================================
ALREADY_SPLIT_FLAG_GROUPS = {
    "behaviour": [
        "Running", "Chasing", "Climbing", "Eating", "Foraging",
    ],
    "signal": [
        "Kuks", "Quaas", "Moans", "Tail flags", "Tail twitches",
    ],
    "highlight": [
        "highlight_gray",
        "highlight_white",
        "highlight_cinnamon",
        "highlight_black",
        "highlight_missing",
    ],
    "animals": [
        "animals_human_present",
        "animals_dog_present",
        "animals_pigeon_present",
        "animals_bird_present",
        "animals_sparrow_present",
        "animals_duck_present",
        "animals_data_missing",
    ],
}

ALREADY_SPLIT_FLAG_COLS = [
    col
    for cols in ALREADY_SPLIT_FLAG_GROUPS.values()
    for col in cols
]

OTHER_FLAGS = ["is_weekend", "is_above_ground", "location_missing"]

BOOLEAN_FLAGS = (
    ALREADY_SPLIT_FLAG_COLS + OTHER_FLAGS
)

ALL_PREDICTOR_COLS = NUMERIC_COLS + CATEGORICAL_COLS + BOOLEAN_FLAGS

# ============================================================
# Hyperparameter grids
# ============================================================
LR_GRID = {"model__C": [0.01, 0.1, 1, 10]}
LR_FIXED = {
    "penalty": "l1",
    "solver": "liblinear",
    "max_iter": 2000,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
}

RF_GRID = {
    'model__max_depth': [3, 5, 7],
    'model__min_samples_leaf': [5, 8, 11],
}
RF_FIXED = {
    "n_estimators": 500,
    "min_samples_split": 20,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# Inner CV scorer + threshold sweep
INNER_CV_SCORER = "balanced_accuracy"
THRESHOLD_RANGE = np.arange(0.05, 0.951, 0.01)

# Bootstrap for test-set CIs
BOOTSTRAP_N = 1000
BOOTSTRAP_LOW_PCT = 2.5
BOOTSTRAP_HIGH_PCT = 97.5

# Permutation importance
PERM_N_REPEATS = 30

experiment_config = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RANDOM_STATE": RANDOM_STATE,
    "TARGET_COL": TARGET_COL,
    "SESSION_COL": SESSION_COL,
    "N_SPLITS_INNER": N_SPLITS_INNER,
    "NUMERIC_COLS": NUMERIC_COLS,
    "OHE_COLS": OHE_COLS,
    "MULTILABEL_COLS": MULTILABEL_COLS,
    "CATEGORICAL_COLS": CATEGORICAL_COLS,
    "CATEGORY_DICT": CATEGORY_DICT,
    "ALREADY_SPLIT_FLAG_GROUPS": ALREADY_SPLIT_FLAG_GROUPS,
    "OTHER_FLAGS": OTHER_FLAGS,
    "BOOLEAN_FLAGS": BOOLEAN_FLAGS,
    "ALL_PREDICTOR_COLS": ALL_PREDICTOR_COLS,
    "LR_GRID": LR_GRID,
    "LR_FIXED": LR_FIXED,
    "RF_GRID": RF_GRID,
    "RF_FIXED": RF_FIXED,
    "INNER_CV_SCORER": INNER_CV_SCORER,
    "THRESHOLD_RANGE": THRESHOLD_RANGE,
    "BOOTSTRAP_N": BOOTSTRAP_N,
    "BOOTSTRAP_LOW_PCT": BOOTSTRAP_LOW_PCT,
    "BOOTSTRAP_HIGH_PCT": BOOTSTRAP_HIGH_PCT,
    "PERM_N_REPEATS": PERM_N_REPEATS,
}

In [91]:
"""Map `Sighter Observed Weather Data` free text to a single 4-category column.

Output values: {sunny, cloudy, rainy, unknown}

Priority order resolves multi-descriptor cells:
  rainy > cloudy > sunny > unknown

Examples:
  '70º F, sunny'             -> sunny
  '65º F, cloudy, drizzle'   -> rainy   (rain > cloud)
  '55º F, partly cloudy'     -> cloudy
  '60º F, breezy'            -> unknown (wind-only is not in the 4-category set)
  '70º F'                    -> unknown

Run as a script for a sanity-check breakdown on data/hectare.csv:
    python scripts/weather_category.py
"""

from __future__ import annotations

import re
from pathlib import Path

import pandas as pd


# Priority-ordered rules: first match wins.
RULES: list[tuple[str, list[str]]] = [
    ("rainy",  [r"\brain", r"\bdrizzl", r"\bsprinkl", r"\bmist", r"\bfog", r"\bshower"]),
    ("cloudy", [r"\bcloud\w*", r"\bovercast\b"]),
    ("sunny",  [r"\bsunny\b", r"\bsun\b", r"\bclear\b", r"\bblue sk"]),
]

# Common typos in the raw data, normalised before pattern matching.
TYPO_FIXES: dict[str, str] = {
    r"\bcoudy\b": "cloudy",
    r"\bsuny\b": "sunny",
}


_COMPILED = [(label, [re.compile(p, re.IGNORECASE) for p in pats]) for label, pats in RULES]
_TYPO = [(re.compile(k, re.IGNORECASE), v) for k, v in TYPO_FIXES.items()]


def _normalise(text: str) -> str:
    for pat, repl in _TYPO:
        text = pat.sub(repl, text)
    return text


def _label(text: object) -> str:
    if not isinstance(text, str):
        return "unknown"
    text = _normalise(text)
    for label, patterns in _COMPILED:
        for p in patterns:
            if p.search(text):
                return label
    return "unknown"


def weather_category(series: pd.Series) -> pd.Series:
    """Return a Series of {sunny, cloudy, rainy, unknown} aligned to ``series``."""
    return series.apply(_label).rename("weather")

In [92]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# from modelling.config import ALL_PREDICTOR_COLS, SESSION_COL, TARGET_COL
# from modelling.weather_category import weather_category

def fix_missing_values(df):
    # above_ground_numeric
    # fill 0 to missing values in above_ground_numeric if location_missing is True
    df.loc[df["location_missing"], "above_ground_numeric"] = df.loc[df["location_missing"], "above_ground_numeric"].fillna(0)
    # extract rows where is_above_ground is True and fill missing values in above_ground_numeric with median
    median_value = df.loc[df["is_above_ground"], "above_ground_numeric"].median()
    df.loc[df["is_above_ground"], "above_ground_numeric"] = df.loc[df["is_above_ground"], "above_ground_numeric"].fillna(median_value)
    assert df["above_ground_numeric"].isna().sum() == 0, "above_ground_numeric still has missing values after imputation"

    # fix Litter missing values by filling "Unknown"
    df["Litter"] = df["Litter"].fillna("Unknown")
    assert df["Litter"].isna().sum() == 0, "Litter still has missing values after imputation"
    return df

def extract_weather_condition(df):
    weather_raw = df["Sighter Observed Weather Data"]
    df['weather_condition'] = weather_category(weather_raw)
    assert df['weather_condition'].isna().sum() == 0, "weather_condition still has missing values after extraction"
    return df

def encode_categorical_columns(df):
    # split the string by comma and strip the whitespace
    df['Hectare Conditions'] = df['Hectare Conditions'].apply(lambda x: [item.strip() for item in x.split(',')])
    # use MultiLabelBinarizer to encode the Hectare Conditions column, named as hectare_condition_<condition>
    mlb = MultiLabelBinarizer()
    hectare_conditions_encoded = mlb.fit_transform(df['Hectare Conditions'])
    hectare_conditions_df = pd.DataFrame(hectare_conditions_encoded, columns=mlb.classes_)
    hectare_conditions_df.columns = [f"hectare_condition_{col}" for col in hectare_conditions_df.columns]
    df = pd.concat([df, hectare_conditions_df], axis=1)
    # drop the original Hectare Conditions column
    df = df.drop(columns=['Hectare Conditions'])

    # these can be one-hot encoded beforehand since they are defined in the description and have a limited number of possible values
    ohe_columns = ['Shift', 'Age', 'Primary Fur Color', 'Litter', 'weather_condition']
    # # check the unique values of these columns to make sure they are suitable for one-hot encoding
    # for col in ohe_columns:
    #     print(f"{col}: {df[col].unique()}")
    # one-hot encode the columns and concatenate with the original dataframe
    df = pd.get_dummies(df, columns=ohe_columns, prefix=ohe_columns)

    return df

def preprocess_loaded_data(df):
    df = fix_missing_values(df)
    df = extract_weather_condition(df)
    # df = encode_categorical_columns(df)
    return df

def select_model_columns(df):
    # select only the columns needed for modelling
    columns = [TARGET_COL, SESSION_COL] + ALL_PREDICTOR_COLS
    missing = [c for c in columns if c not in df.columns]
    if missing:
        print(f"WARNING: missing columns ({len(missing)}): {missing}")
    df = df[columns]
    return df

def select_response_subset(df):
    # select only the rows with response values in [0, 1]
    print(f"Original class balance: {df[TARGET_COL].value_counts(dropna=False).to_dict()}")
    df = df[df[TARGET_COL].isin([0, 1])]
    print(f"Filtered class balance: {df[TARGET_COL].value_counts(dropna=False).to_dict()}")
    return df

def load_processed_data(path=None):
    # load either parquet or csv depending on the file extension
    if path.suffix == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file format: {path.suffix}")

    df = preprocess_loaded_data(df)
    df = select_model_columns(df)
    df = select_response_subset(df)

    print(f"Loaded {len(df)} rows x {df.shape[1]} columns from {path}")

    return df


In [93]:
from dataclasses import dataclass

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# from modelling.config import ALL_PREDICTOR_COLS, RANDOM_STATE, SESSION_COL, TARGET_COL


@dataclass
class SplitResult:
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    groups_train: pd.Series
    groups_test: pd.Series


def stratified_grouped_split(df):
    """80/20 stratified-grouped hold-out via the first fold of StratifiedGroupKFold(5)."""
    feature_cols = [c for c in ALL_PREDICTOR_COLS if c in df.columns]

    X = df[feature_cols].copy()
    y = df[TARGET_COL].astype(int)
    groups = df[SESSION_COL]

    # grouped by session_id, so that all rows from a session are in the same split. prevent leakage of session-specific info (e.g. location). stratified by target, to maintain class balance in both splits.
    stratified_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    train_idx, test_idx = next(stratified_folds.split(X, y, groups=groups))

    s = SplitResult(
        X_train=X.iloc[train_idx].reset_index(drop=True),
        X_test=X.iloc[test_idx].reset_index(drop=True),
        y_train=y.iloc[train_idx].reset_index(drop=True),
        y_test=y.iloc[test_idx].reset_index(drop=True),
        groups_train=groups.iloc[train_idx].reset_index(drop=True),
        groups_test=groups.iloc[test_idx].reset_index(drop=True),
    )

    overlap = set(s.groups_train.unique()) & set(s.groups_test.unique())
    if overlap:
        raise RuntimeError(f"Session leakage: {len(overlap)} sessions in both splits")

    n_total = len(s.y_train) + len(s.y_test)
    print(f"Train: {len(s.y_train)} rows ({100 * len(s.y_train) / n_total:.1f}%)")
    print(f"Test:  {len(s.y_test)} rows ({100 * len(s.y_test) / n_total:.1f}%)")
    print(f"Train class balance: {s.y_train.value_counts().to_dict()}")
    print(f"Test class balance:  {s.y_test.value_counts().to_dict()}")
    print(f"Sessions: train={s.groups_train.nunique()} unique, test={s.groups_test.nunique()} unique")

    return s


In [94]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
import numpy as np
import pandas as pd

# from modelling.config import (
#     BOOLEAN_FLAGS, CATEGORY_DICT, OHE_COLS, LR_FIXED, NUMERIC_COLS,
#     RANDOM_STATE, RF_FIXED,
# )


def resolve_columns(df):
    num = [c for c in NUMERIC_COLS if c in df.columns]
    ohe = [c for c in OHE_COLS if c in df.columns]
    hectare = "Hectare Conditions" if "Hectare Conditions" in df.columns else None
    flag = [c for c in BOOLEAN_FLAGS if c in df.columns]
    return num, ohe, hectare, flag


def numeric_branch(scale):
    if scale:
        return Pipeline([("scaler", StandardScaler())])
    return "passthrough"


def ohe_branch(ohe_cols):
    ohe = OneHotEncoder(
                    categories=[CATEGORY_DICT[c] for c in ohe_cols],
                    handle_unknown="ignore",
                    sparse_output=False,
                )
    return Pipeline([("ohe", ohe)])


def encode_hectare_conditions(X):
    """
    X: DataFrame with one column: 'Hectare Conditions'
       value examples:
       - "Busy, Calm"
       - "Moderate"
       - np.nan
       - ["Busy", "Calm"]  # also allowed
    return: DataFrame with fixed columns:
       hectare_condition_Busy / Calm / Moderate
    """
    col = X.iloc[:, 0]

    allowed = CATEGORY_DICT["Hectare Conditions"]
    out = pd.DataFrame(
        0,
        index=X.index,
        columns=[f"hectare_condition_{c}" for c in allowed],
    )

    for idx, value in col.items():
        if isinstance(value, list):
            labels = value
        elif pd.isna(value):
            labels = []
        else:
            labels = [item.strip() for item in str(value).split(",")]

        for label in labels:
            if label in allowed:
                out.loc[idx, f"hectare_condition_{label}"] = 1

    return out


def hectare_branch():
    hectare_transformer = FunctionTransformer(
        encode_hectare_conditions,
        validate=False,
        feature_names_out=lambda self, input_features: np.array(
            [f"hectare_condition_{c}" for c in CATEGORY_DICT["Hectare Conditions"]]
        ),
    )
    return Pipeline([
        ("hectare", hectare_transformer),
    ])

def build_lr_pipeline(df):
    num, ohe, hec, flag = resolve_columns(df)
    pre = ColumnTransformer(
        transformers=[
            ("num", numeric_branch(scale=True), num),
            ("ohe", ohe_branch(ohe), ohe),
            ("hectare", hectare_branch(), [hec] if hec else []),
            ("flag", "passthrough", flag)
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    return Pipeline([("pre", pre), ("model", LogisticRegression(**LR_FIXED))])


def build_rf_pipeline(df):
    num, ohe, hec, flag = resolve_columns(df)
    pre = ColumnTransformer(
        transformers=[
            ("num", numeric_branch(scale=False), num),
            ("ohe", ohe_branch(ohe), ohe),
            ("hectare", hectare_branch(), [hec] if hec else []),
            ("flag", "passthrough", flag)
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    return Pipeline([("pre", pre), ("model", RandomForestClassifier(**RF_FIXED))])


def build_dummy_pipeline():
    return Pipeline([
        ("model", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
    ])


def get_feature_names(pipeline):
    pre = pipeline.named_steps.get("pre")
    if pre is None:
        return []
    return list(pre.get_feature_names_out())


In [95]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)

# from modelling.config import PROJECT_ROOT


def _relative_path(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT)
    except ValueError:
        return path


def plot_pr_curve(y_true, proba, selected_threshold, name, output_dir):
    """Save an out-of-fold PR curve and mark the selected threshold."""
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)

    precision, recall, _ = precision_recall_curve(y_true, proba)
    average_precision = average_precision_score(y_true, proba)

    selected_pred = (proba >= selected_threshold).astype(int)
    selected_precision = precision_score(
        y_true, selected_pred, pos_label=1, zero_division=0
    )
    selected_recall = recall_score(
        y_true, selected_pred, pos_label=1, zero_division=0
    )

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"pr_curve_{name.lower()}.png"

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(
        recall,
        precision,
        linewidth=1.8,
        label=f"OOF PR curve (AP = {average_precision:.3f})",
    )
    ax.axhline(
        y_true.mean(),
        color="0.45",
        linestyle=":",
        linewidth=1.2,
        label="positive-class baseline",
    )
    ax.scatter(
        selected_recall,
        selected_precision,
        color="tab:red",
        zorder=3,
        label=f"selected threshold = {selected_threshold:.2f}",
    )

    ax.set_title(f"{name}: out-of-fold PR curve")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(axis="both", alpha=0.25)
    ax.legend(frameon=False)

    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)

    print(f"Wrote {_relative_path(out)}")
    return out


In [96]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_val_predict

# from modelling.config import (
#     INNER_CV_SCORER, LR_GRID, N_SPLITS_INNER, RANDOM_STATE,
#     RF_GRID, THRESHOLD_RANGE,
# )
# from modelling.threshold_plots import plot_pr_curve


@dataclass
class TunedModel:
    name: str
    pipeline: object
    best_params: dict
    cv_results: pd.DataFrame
    threshold: float


def inner_cv():
    return StratifiedGroupKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)


def grid_search(pipeline, grid, X, y, groups, name):
    print(f"[{name}] GridSearchCV  grid={grid}  scorer={INNER_CV_SCORER}")
    gs = GridSearchCV(
        pipeline,
        param_grid=grid,
        scoring=INNER_CV_SCORER,
        cv=inner_cv(),
        refit=True,
        n_jobs=-1,
        return_train_score=False,
    )
    gs.fit(X, y, groups=groups)
    cv_df = pd.DataFrame(gs.cv_results_)
    print(f"[{name}] best_params={gs.best_params_}  best_inner_CV_AP={gs.best_score_:.4f}")
    return gs.best_estimator_, gs.best_params_, cv_df


def tune_threshold(pipeline, X, y, groups, name, output_dir=None):
    proba_oof = cross_val_predict(
        pipeline, X, y,
        groups=groups,
        cv=inner_cv(),
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]
    scores = [
        f1_score(y, (proba_oof >= t).astype(int), average="macro", zero_division=0)
        for t in THRESHOLD_RANGE
    ]
    best_idx = int(np.argmax(scores))
    best_t = float(THRESHOLD_RANGE[best_idx])
    if output_dir is not None:
        plot_pr_curve(y, proba_oof, best_t, name, output_dir)
    print(f"[{name}] best_threshold={best_t:.2f}  train OOF macro-F1={scores[best_idx]:.4f}")
    return best_t


def tune_model(pipeline, grid, X, y, groups, name, output_dir=None):
    # inner cv for hyperparameter tuning, then refit on the whole training set
    refit_pipeline, best_params, cv_df = grid_search(pipeline, grid, X, y, groups, name)
    # inner cv for threshold tuning, using the refitted pipeline to get OOF probabilities
    threshold = tune_threshold(refit_pipeline, X, y, groups, name, output_dir=output_dir)
    return TunedModel(
        name=name,
        pipeline=refit_pipeline,
        best_params=best_params,
        cv_results=cv_df,
        threshold=0.5, # TEMP: use 0.5 for now to simplify the evaluation and comparison later, since we are mainly interested in the relative performance of the models rather than the absolute performance. can add threshold tuning back later if we have time.
    )


def fit_dummy(pipeline, X, y):
    pipeline.fit(X, y)
    return TunedModel(
        name="Dummy",
        pipeline=pipeline,
        best_params={},
        cv_results=pd.DataFrame(),
        threshold=0.5,
    )


def tune_all(split, build_lr, build_rf, build_dummy, output_dir=None):
    """Tune dummy / LR / RF on the training portion of an outer split."""
    tuned = {}

    print("--- Dummy (most_frequent) ---")
    tuned["Dummy"] = fit_dummy(build_dummy(), split.X_train, split.y_train)

    print("--- Logistic Regression ---")
    tuned["LR"] = tune_model(
        build_lr(split.X_train), LR_GRID,
        split.X_train, split.y_train, split.groups_train,
        name="LR",
        output_dir=output_dir,
    )

    print("--- Random Forest ---")
    tuned["RF"] = tune_model(
        build_rf(split.X_train), RF_GRID,
        split.X_train, split.y_train, split.groups_train,
        name="RF",
        output_dir=output_dir,
    )

    return tuned


In [97]:
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# from modelling.config import (
#     BOOTSTRAP_HIGH_PCT, BOOTSTRAP_LOW_PCT, BOOTSTRAP_N, RANDOM_STATE,
# )


METRIC_KEYS = [
    "accuracy", "macro_f1", "balanced_accuracy",
    "precision_approach", "recall_approach", "pr_auc",
]


@dataclass
class MetricResult:
    point: float
    ci_low: float = float("nan")
    ci_high: float = float("nan")


@dataclass
class EvaluationResult:
    name: str
    threshold: float
    test_metrics: dict = field(default_factory=dict)
    train_metrics: dict = field(default_factory=dict)
    confusion: np.ndarray = None


def compute_metrics(y_true, y_proba, y_pred):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_approach": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_approach": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }
    if y_proba is not None and len(np.unique(y_true)) > 1:
        out["pr_auc"] = average_precision_score(y_true, y_proba)
    else:
        out["pr_auc"] = float("nan")
    return out


def predict(pipeline, X, threshold):
    if hasattr(pipeline, "predict_proba"):
        proba = pipeline.predict_proba(X)[:, 1]
        preds = (proba >= threshold).astype(int)
        return proba, preds
    return None, np.asarray(pipeline.predict(X), dtype=int)


def bootstrap_indices(n, n_resamples, seed):
    rng = np.random.default_rng(seed)
    return rng.integers(0, n, size=(n_resamples, n))


def bootstrap_metrics(y_true, y_proba, y_pred, indices):
    samples = {k: [] for k in METRIC_KEYS}

    for i in range(indices.shape[0]):
        idx = indices[i]
        yt = y_true[idx]
        yp = y_pred[idx]
        ypr = y_proba[idx] if y_proba is not None else None
        if len(np.unique(yt)) < 2: # skip when samples doesn't contain both classes
            continue
        m = compute_metrics(yt, ypr, yp)
        for k in METRIC_KEYS:
            samples[k].append(m[k])

    # point estimate on test set
    point = compute_metrics(y_true, y_proba, y_pred)

    out = {}
    for k in METRIC_KEYS:
        arr = np.array(samples[k], dtype=float)
        arr = arr[~np.isnan(arr)]
        if len(arr) == 0:
            out[k] = MetricResult(point=point[k])
        else:
            out[k] = MetricResult(
                point=point[k],
                ci_low=float(np.percentile(arr, BOOTSTRAP_LOW_PCT)),
                ci_high=float(np.percentile(arr, BOOTSTRAP_HIGH_PCT)),
            )
    return out


def evaluate(name, pipeline, threshold, X_train, y_train, X_test, y_test):
    y_train_arr = y_train.to_numpy(dtype=int)
    y_test_arr = y_test.to_numpy(dtype=int)

    proba_test, pred_test = predict(pipeline, X_test, threshold)
    proba_train, pred_train = predict(pipeline, X_train, threshold)

    indices = bootstrap_indices(len(y_test_arr), BOOTSTRAP_N, seed=RANDOM_STATE)
    test_metrics = bootstrap_metrics(y_test_arr, proba_test, pred_test, indices)

    train_point = compute_metrics(y_train_arr, proba_train, pred_train)
    train_metrics = {k: MetricResult(point=v) for k, v in train_point.items()}

    cm = confusion_matrix(y_test_arr, pred_test, labels=[0, 1])

    print(
        f"[{name}] threshold={threshold:.2f}  "
        f"TEST  macro-F1={test_metrics['macro_f1'].point:.3f} "
        f"[{test_metrics['macro_f1'].ci_low:.3f}, {test_metrics['macro_f1'].ci_high:.3f}]  "
        f"bal-acc={test_metrics['balanced_accuracy'].point:.3f}  "
        f"PR-AUC={test_metrics['pr_auc'].point:.3f}"
    )
    print(
        f"[{name}] threshold={threshold:.2f}  "
        f"TRAIN macro-F1={train_metrics['macro_f1'].point:.3f}  "
        f"bal-acc={train_metrics['balanced_accuracy'].point:.3f}  "
        f"PR-AUC={train_metrics['pr_auc'].point:.3f}"
    )
    print(f"[{name}] confusion matrix (rows=true, cols=pred, [0,1]):\n{cm}")

    return EvaluationResult(
        name=name,
        threshold=threshold,
        test_metrics=test_metrics,
        train_metrics=train_metrics,
        confusion=cm,
    )


def evaluate_all(tuned, split):
    return [
        evaluate(
            name=tm.name,
            pipeline=tm.pipeline,
            threshold=tm.threshold,
            X_train=split.X_train,
            y_train=split.y_train,
            X_test=split.X_test,
            y_test=split.y_test,
        )
        for tm in tuned.values()
    ]


def metrics_to_frame(results):
    rows = []
    for r in results:
        for split_name, store in (("test", r.test_metrics), ("train", r.train_metrics)):
            for metric, mr in store.items():
                rows.append({
                    "model": r.name,
                    "split": split_name,
                    "metric": metric,
                    "point": mr.point,
                    "ci_low": mr.ci_low,
                    "ci_high": mr.ci_high,
                    "threshold": r.threshold,
                })
    return pd.DataFrame(rows)


In [98]:
import pandas as pd
from sklearn.inspection import permutation_importance

# from modelling.config import (
#     CATEGORICAL_COLS, INNER_CV_SCORER, PERM_N_REPEATS, RANDOM_STATE,
# )
# from modelling.pipelines import get_feature_names


def aggregate_to_parent(transformed_names, values, raw_categorical, aggfunc):
    """Sum OHE dummies back to the raw (parent) feature for cross-model comparison."""
    parents = []
    for name in transformed_names:
        rest = name.split("__", 1)[1] if "__" in name else name
        matched = None
        for cat in raw_categorical:
            if rest == cat or rest.startswith(cat + "_"):
                matched = cat
                break
        parents.append(matched or rest)

    df_out = pd.DataFrame({"feature": parents, "value": values})
    if aggfunc == "sum_abs":
        return df_out.assign(value=df_out["value"].abs()).groupby("feature", as_index=False)["value"].sum()
    return df_out.groupby("feature", as_index=False)["value"].sum()


def lr_coefficients(pipeline, raw_categorical):
    model = pipeline.named_steps["model"]
    feat_names = get_feature_names(pipeline)
    coefs = model.coef_.ravel()

    per_dummy = pd.DataFrame({"feature": feat_names, "coefficient": coefs})

    aggregated = aggregate_to_parent(feat_names, coefs, raw_categorical, "sum_abs")
    aggregated = aggregated.rename(columns={"value": "abs_coefficient_sum"})
    aggregated = aggregated.sort_values("abs_coefficient_sum", ascending=False).reset_index(drop=True)

    print("LR top-10 features (sum |coef|):")
    print(aggregated.head(10).to_string(index=False))
    return aggregated, per_dummy


def rf_impurity_importance(pipeline, raw_categorical):
    model = pipeline.named_steps["model"]
    feat_names = get_feature_names(pipeline)
    importances = model.feature_importances_

    aggregated = aggregate_to_parent(feat_names, importances, raw_categorical, "sum")
    aggregated = aggregated.rename(columns={"value": "impurity_importance"})
    aggregated = aggregated.sort_values("impurity_importance", ascending=False).reset_index(drop=True)

    print("RF impurity top-10 features:")
    print(aggregated.head(10).to_string(index=False))
    return aggregated


def model_permutation_importance(name, pipeline, X_test, y_test):
    print(f"{name} permutation importance: n_repeats={PERM_N_REPEATS}, scoring={INNER_CV_SCORER}")
    result = permutation_importance(
        pipeline, X_test, y_test,
        n_repeats=PERM_N_REPEATS,
        scoring=INNER_CV_SCORER,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    df_perm = pd.DataFrame({
        "feature": X_test.columns,
        "perm_importance_mean": result.importances_mean,
        "perm_importance_std": result.importances_std,
    }).sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)

    print(f"{name} permutation top-10 features:")
    print(df_perm.head(10).to_string(index=False))
    return df_perm


def analyse_all(tuned, split):
    raw_cat = [c for c in CATEGORICAL_COLS if c in split.X_train.columns]

    lr_agg, lr_per_dummy = lr_coefficients(tuned["LR"].pipeline, raw_cat)
    rf_imp = rf_impurity_importance(tuned["RF"].pipeline, raw_cat)
    lr_perm = model_permutation_importance("LR", tuned["LR"].pipeline, split.X_test, split.y_test)
    rf_perm = model_permutation_importance("RF", tuned["RF"].pipeline, split.X_test, split.y_test)

    return {
        "lr_coefficients_aggregated": lr_agg,
        "lr_coefficients_per_dummy": lr_per_dummy,
        "lr_permutation": lr_perm,
        "rf_impurity": rf_imp,
        "rf_permutation": rf_perm,
    }


In [99]:
from datetime import datetime
from pathlib import Path
from pprint import pformat

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# from modelling import config as experiment_config
# from modelling.config import OUTPUT_DIR, PROJECT_ROOT


CONFIG_SECTIONS = {
    "Paths": ["PROJECT_ROOT", "OUTPUT_DIR"],
    "Reproducibility": ["RANDOM_STATE"],
    "Target / split": ["TARGET_COL", "SESSION_COL", "N_SPLITS_INNER"],
    "Predictors": [
        "NUMERIC_COLS",
        "OHE_COLS",
        "MULTILABEL_COLS",
        "CATEGORICAL_COLS",
        "CATEGORY_DICT",
        "ALREADY_SPLIT_FLAG_GROUPS",
        "OTHER_FLAGS",
        "BOOLEAN_FLAGS",
        "ALL_PREDICTOR_COLS",
    ],
    "Hyperparameters": ["LR_GRID", "LR_FIXED", "RF_GRID", "RF_FIXED"],
    "Scoring / thresholding": ["INNER_CV_SCORER", "THRESHOLD_RANGE"],
    "Bootstrap": ["BOOTSTRAP_N", "BOOTSTRAP_LOW_PCT", "BOOTSTRAP_HIGH_PCT"],
    "Permutation importance": ["PERM_N_REPEATS"],
}


def create_experiment_dir():
    des_dir = OUTPUT_DIR / f"experiment_{datetime.now().astimezone().strftime('%Y%m%d_%H%M%S')}"
    des_dir.mkdir(parents=True, exist_ok=True)
    return des_dir


def relative_path(path):
    path = Path(path)
    if path.is_absolute():
        try:
            return path.relative_to(PROJECT_ROOT)
        except ValueError:
            return path
    return path


def format_config_value(value):
    if isinstance(value, Path):
        return str(relative_path(value))
    if isinstance(value, np.ndarray):
        if value.size == 0:
            return "[]"
        if value.size == 1:
            return pformat(value.tolist(), width=100, sort_dicts=False)
        step = value[1] - value[0]
        return (
            f"{value[0]:.4g} to {value[-1]:.4g} "
            f"(n={value.size}, step={step:.4g})"
        )
    return pformat(value, width=100, sort_dicts=False)


def write_experiment_config(tuned=None, data_path=None, des_dir=None, path=None):
    if path is not None:
        out = Path(path)
    else:
        out_dir = Path(des_dir) if des_dir is not None else create_experiment_dir()
        out = out_dir / "experiment_config.txt"
    out.parent.mkdir(parents=True, exist_ok=True)

    lines = [
        "Experiment configuration",
        f"Generated at: {datetime.now().astimezone().isoformat(timespec='seconds')}",
        f"Experiment directory: {relative_path(out.parent)}",
    ]
    if data_path is not None:
        lines.append(f"Data path: {relative_path(data_path)}")

    for section_name, names in CONFIG_SECTIONS.items():
        lines.extend(["", section_name, "-" * len(section_name)])
        for name in names:
            value = experiment_config.get(name, "<undefined>")
            lines.append(f"{name}: {format_config_value(value)}")

    if tuned:
        section_name = "Tuned model settings"
        lines.extend(["", section_name, "-" * len(section_name)])
        for name, tm in tuned.items():
            lines.append(f"{name}:")
            lines.append(f"  threshold: {tm.threshold:.4g}")
            lines.append(f"  best_params: {format_config_value(tm.best_params)}")

    out.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Wrote {relative_path(out)}")


def save_csv(df_out, path, index=False):
    df_out.to_csv(path, index=index)
    rel = path.relative_to(PROJECT_ROOT) if path.is_absolute() else path
    print(f"Wrote {rel} ({len(df_out)} rows)")


def save_metrics(metrics_df, des_dir):
    save_csv(metrics_df, des_dir / "metrics_summary.csv")


def plot_metrics_summary(metrics_df, des_dir, path=None):
    metric_order = [
        "accuracy",
        "macro_f1",
        "balanced_accuracy",
        "precision_approach",
        "recall_approach",
        "pr_auc",
    ]
    metric_labels = {
        "accuracy": "Accuracy",
        "macro_f1": "Macro F1",
        "balanced_accuracy": "Balanced accuracy",
        "precision_approach": "Precision (approach)",
        "recall_approach": "Recall (approach)",
        "pr_auc": "PR AUC",
    }

    metrics = [m for m in metric_order if m in metrics_df["metric"].unique()]
    if not metrics:
        return

    models = metrics_df["model"].drop_duplicates().tolist()
    x = np.arange(len(models))
    width = 0.36

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
    axes = axes.ravel()
    colors = {"train": "tab:blue", "test": "tab:orange"}

    for ax, metric in zip(axes, metrics):
        metric_df = metrics_df[metrics_df["metric"] == metric]
        y_max = 1.0

        for split_name, offset in (("train", -width / 2), ("test", width / 2)):
            split_df = (
                metric_df[metric_df["split"] == split_name]
                .set_index("model")
                .reindex(models)
            )
            points = split_df["point"].to_numpy(dtype=float)

            yerr = None
            if split_name == "test":
                lows = split_df["ci_low"].to_numpy(dtype=float)
                highs = split_df["ci_high"].to_numpy(dtype=float)
                valid_ci = ~np.isnan(lows) & ~np.isnan(highs)
                lower_err = np.where(valid_ci, np.maximum(points - lows, 0), 0)
                upper_err = np.where(valid_ci, np.maximum(highs - points, 0), 0)
                yerr = np.vstack([lower_err, upper_err])
                if np.any(valid_ci):
                    y_max = max(y_max, np.nanmax(highs))

            bars = ax.bar(
                x + offset,
                points,
                width,
                label=split_name.title(),
                color=colors[split_name],
                yerr=yerr,
                capsize=4 if yerr is not None else 0,
                error_kw={"elinewidth": 1, "capthick": 1},
            )
            y_max = max(y_max, np.nanmax(points))

            for i, bar in enumerate(bars):
                point = points[i]
                if np.isnan(point):
                    continue
                text_y = point
                ax.annotate(
                    f"{point:.2f}",
                    xy=(bar.get_x() + bar.get_width() / 2, text_y),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

        ax.set_title(metric_labels.get(metric, metric))
        ax.set_xticks(x, models)
        ax.set_ylim(0, min(1.15, y_max + 0.12))
        ax.grid(axis="y", alpha=0.25)

    for ax in axes[len(metrics):]:
        ax.axis("off")

    axes[0].set_ylabel("Score")
    axes[3].set_ylabel("Score")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.94))

    out = Path(path) if path is not None else Path(des_dir) / "metrics_summary.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Wrote {relative_path(out)}")


def save_confusion_matrices(eval_results, des_dir):
    for r in eval_results:
        if r.confusion is None:
            continue
        cm_df = pd.DataFrame(
            r.confusion,
            index=["true_avoid", "true_approach"],
            columns=["pred_avoid", "pred_approach"],
        )
        save_csv(cm_df, des_dir / f"confusion_matrix_{r.name.lower()}.csv", index=True)


def save_grid_search_tables(tuned, des_dir):
    for name, tm in tuned.items():
        if tm.cv_results.empty:
            continue
        keep = [c for c in tm.cv_results.columns
                if c.startswith("param_") or c in ("mean_test_score", "std_test_score", "rank_test_score")]
        save_csv(tm.cv_results[keep], des_dir / f"grid_search_{name.lower()}.csv")


def save_feature_influence(influence, des_dir):
    for tag, df_out in influence.items():
        save_csv(df_out, des_dir / f"feature_influence_{tag}.csv")


def plot_confusion_matrices(eval_results, des_dir, path=None):
    fig, axes = plt.subplots(1, len(eval_results), figsize=(4 * len(eval_results), 3.5))
    if len(eval_results) == 1:
        axes = [axes]
    for ax, r in zip(axes, eval_results):
        if r.confusion is None:
            continue
        ax.imshow(r.confusion, cmap="Blues")
        ax.set_title(f"{r.name} (threshold={r.threshold:.2f})")
        ax.set_xticks([0, 1], ["pred avoid", "pred approach"])
        ax.set_yticks([0, 1], ["true avoid", "true approach"])
        for i in range(2):
            for j in range(2):
                ax.text(
                    j, i, r.confusion[i, j],
                    ha="center", va="center",
                    color="white" if r.confusion[i, j] > r.confusion.max() / 2 else "black",
                )
    fig.tight_layout()
    out = Path(path) if path is not None else Path(des_dir) / "confusion_matrices.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Wrote {relative_path(out)}")


def plot_feature_influence(influence, des_dir, top_n=15, path=None):
    lr_top = influence["lr_coefficients_aggregated"].head(top_n).iloc[::-1]
    rf_imp_top = influence["rf_impurity"].head(top_n).iloc[::-1]
    lr_perm_top = influence["lr_permutation"].head(top_n).iloc[::-1]
    rf_top = influence["rf_permutation"].head(top_n).iloc[::-1]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ax1, ax2, ax3, ax4 = axes.ravel()

    ax1.barh(lr_top["feature"], lr_top["abs_coefficient_sum"], color="tab:blue")
    ax1.set_title(f"LR  top {top_n} by sum |coef|")
    ax1.set_xlabel("Sum of |standardised coefficients|")

    ax2.barh(
        rf_imp_top["feature"], rf_imp_top["impurity_importance"],
        color="tab:purple",
    )
    ax2.set_title(f"RF  top {top_n} impurity importance")
    ax2.set_xlabel("Mean decrease in impurity")

    ax3.barh(
        lr_perm_top["feature"], lr_perm_top["perm_importance_mean"],
        color="tab:orange", xerr=lr_perm_top["perm_importance_std"],
    )
    ax3.set_title(f"LR  top {top_n} permutation importance")
    ax3.set_xlabel(f"Mean drop in {experiment_config['INNER_CV_SCORER']}")

    ax4.barh(
        rf_top["feature"], rf_top["perm_importance_mean"],
        color="tab:green", xerr=rf_top["perm_importance_std"],
    )
    ax4.set_title(f"RF  top {top_n} permutation importance")
    ax4.set_xlabel(f"Mean drop in {experiment_config['INNER_CV_SCORER']}")

    fig.tight_layout()
    out = Path(path) if path is not None else Path(des_dir) / "feature_influence_compare.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Wrote {relative_path(out)}")


def write_outputs(tuned, eval_results, influence, metrics_df, data_path=None, des_dir=None):
    if des_dir is None:
        des_dir = create_experiment_dir()
    print(f"Saving outputs to {relative_path(des_dir)}")

    write_experiment_config(tuned=tuned, data_path=data_path, des_dir=des_dir)
    save_metrics(metrics_df, des_dir)
    save_confusion_matrices(eval_results, des_dir)
    save_grid_search_tables(tuned, des_dir)
    save_feature_influence(influence, des_dir)

    plot_metrics_summary(metrics_df, des_dir)
    plot_confusion_matrices(eval_results, des_dir)
    plot_feature_influence(influence, des_dir)


In [100]:
"""Run the squirrel approach/avoid modelling pipeline end to end.

Usage:

    python run_modelling.py
    python run_modelling.py --data data/cleaned_table_2.parquet

Outputs are written under a timestamped outputs/experiment_YYYYMMDD_HHMMSS/ directory.
"""

import argparse
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module=r"sklearn\..*")

# from modelling.data import load_processed_data
# from modelling.evaluation import evaluate_all, metrics_to_frame
# from modelling.feature_influence import analyse_all
# from modelling.outputs import create_experiment_dir, relative_path, write_outputs
# from modelling.pipelines import build_dummy_pipeline, build_lr_pipeline, build_rf_pipeline
# from modelling.split import stratified_grouped_split
# from modelling.tuning import tune_all


def parse_args():
    parser = argparse.ArgumentParser(description="Run the squirrel modelling pipeline.")
    parser.add_argument(
        "--data", "-d",
        type=Path,
        default="data/cleaned_table_2.parquet",
        help="Path to the processed parquet file. Defaults to data/cleaned_table_2.parquet at the project root.",
    )
    return parser.parse_args()


def run(data_path=None):
    print("=" * 60)
    print("LOAD DATA")
    print("=" * 60)
    df = load_processed_data(data_path)

    print("=" * 60)
    print("OUTER 80/20 SPLIT")
    print("=" * 60)
    split = stratified_grouped_split(df)

    des_dir = create_experiment_dir()
    print(f"Experiment outputs: {relative_path(des_dir)}")

    print("=" * 60)
    print("TUNING: DUMMY / LR / RF")
    print("=" * 60)
    tuned = tune_all(
        split,
        build_lr=build_lr_pipeline,
        build_rf=build_rf_pipeline,
        build_dummy=build_dummy_pipeline,
        output_dir=des_dir,
    )

    print("=" * 60)
    print("EVALUATION ON TEST SET")
    print("=" * 60)
    eval_results = evaluate_all(tuned, split)
    metrics_df = metrics_to_frame(eval_results)

    print("=" * 60)
    print("FEATURE INFLUENCE")
    print("=" * 60)
    influence = analyse_all(tuned, split)

    print("=" * 60)
    print("WRITE OUTPUTS")
    print("=" * 60)
    write_outputs(tuned, eval_results, influence, metrics_df, data_path=data_path, des_dir=des_dir)


# ========= MAIN ENTRY POINT =========
start_time = time.time()

# args = parse_args()
run(Path("/Users/zutomayo/workspace/python/comp20008-a2/data/cleaned_table_2.parquet"))

elapsed = time.time() - start_time
print(f"\nTotal elapsed time: {elapsed:.2f} seconds")


LOAD DATA
Original class balance: {nan: 2204, 0.0: 656, 1.0: 158}
Filtered class balance: {0.0: 656, 1.0: 158}
Loaded 814 rows x 41 columns from /Users/zutomayo/workspace/python/comp20008-a2/data/cleaned_table_2.parquet
OUTER 80/20 SPLIT
Train: 660 rows (81.1%)
Test:  154 rows (18.9%)
Train class balance: {0: 527, 1: 133}
Test class balance:  {0: 129, 1: 25}
Sessions: train=309 unique, test=71 unique
Experiment outputs: outputs/experiment_20260511_234344
TUNING: DUMMY / LR / RF
--- Dummy (most_frequent) ---
--- Logistic Regression ---
[LR] GridSearchCV  grid={'model__C': [0.01, 0.1, 1, 10]}  scorer=balanced_accuracy
[LR] best_params={'model__C': 1}  best_inner_CV_AP=0.6761
Wrote outputs/experiment_20260511_234344/pr_curve_lr.png
[LR] best_threshold=0.56  train OOF macro-F1=0.6505
--- Random Forest ---
[RF] GridSearchCV  grid={'model__max_depth': [3, 5, 7], 'model__min_samples_leaf': [5, 8, 11]}  scorer=balanced_accuracy
[RF] best_params={'model__max_depth': 5, 'model__min_samples_leaf'